# Plot SAE Training Metrics
Reads every `results/sae_metrics*.csv` and plots per-layer L0 and MSE,
one line per (model, hook_point, l1_coeff) so different L1 sweeps overlay.
Runs locally, no GPU needed. Update the CSVs after each run, then re-run this.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

files = sorted(glob.glob('results/sae_metrics*.csv'))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df = df.dropna(subset=['l0', 'mse_loss']).sort_values(['l1_coeff', 'layer'])
print('loaded:', files)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for (model, hook, l1), g in df.groupby(['model', 'hook_point', 'l1_coeff']):
    label = f'{hook} / L1={l1:g}'
    axes[0].plot(g['layer'], g['l0'], marker='o', label=label)
    axes[1].plot(g['layer'], g['mse_loss'], marker='o', label=label)

axes[0].axhspan(10, 60, color='green', alpha=0.08, label='healthy L0 band')
axes[0].set_title('L0 (avg active features / patch)')
axes[0].set_xlabel('layer'); axes[0].set_ylabel('L0'); axes[0].grid(alpha=0.3)
axes[1].set_title('MSE loss (raw — not comparable across layers)')
axes[1].set_xlabel('layer'); axes[1].set_ylabel('MSE'); axes[1].grid(alpha=0.3)
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('results/sae_metrics.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved results/sae_metrics.png')